# Stage 3 — download_fulltext

Re-create this stage's script with Gemini's help. The cells below give you the spec, the seed, the gotchas, and a verification step. The implementation itself is yours to write.


## 1. Setup

Every cell in this section is idempotent and safe to re-run. If you opened this notebook fresh (without running Stage 0 first in the same runtime), run all of them now.


### 1a. Clone the repo and `cd` into it


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


### 1b. Install dependencies

Python (`openai`) and the Node CLI `@llamaindex/liteparse`. First run takes ~30s; re-runs are near-instant.


In [ ]:
# Install dependencies. Idempotent (pip skips already-installed; npm re-link is cheap).
# liteparse only matters for Stage 4 but installing it everywhere keeps each
# notebook self-contained, which is the whole point of re-running this cell.
!pip install -q -r requirements.txt
!npm install -g @llamaindex/liteparse 2>&1 | tail -3


### 1c. (no API key needed for this stage)


In [ ]:
# This stage doesn't call the OpenAI API.


### 1d. Stage the bundle-shipped configs


In [ ]:
# Copy bundle-shipped configs into the directories each stage script expects.
# Each stage's input.txt / criteria.txt / schema.json lives under configs/
# in the repo; the actual scripts read them relative to cwd.
import os, shutil
os.makedirs("stage_01", exist_ok=True)
os.makedirs("stage_02", exist_ok=True)
shutil.copy("configs/stage_01_input.txt",    "stage_01/input.txt")
shutil.copy("configs/stage_02_input.txt",    "stage_02/input.txt")
shutil.copy("configs/stage_02_criteria.txt", "stage_02/criteria.txt")
shutil.copy("configs/schema.json",           "schema.json")
print("configs staged")


### 1e. Load prior stages' reference outputs

Stage 3 reads outputs from earlier stages. Each Colab notebook gets its own runtime, so work done in another notebook is not visible here. This cell seeds `stage_01..stage_02/data/` from the canonical reference run so Stage 3 has inputs to work with.


In [ ]:
# Load prior stages' reference outputs as inputs for Stage 3.
# Each Colab notebook opens with a fresh runtime, so any work done in a
# Stage <3 notebook in a DIFFERENT runtime is not visible here.
# This cell makes the stage runnable in isolation against the canonical
# reference run. If you re-run an earlier stage IN THIS runtime, your
# output replaces these reference files (cwd is /content/...).
import os, shutil, glob
for n in range(1, 3):
    dst = f"stage_0{n}/data"
    src = f"reference_outputs/stage_0{n}/data"
    if not os.path.isdir(src):
        continue
    os.makedirs(dst, exist_ok=True)
    # Only seed if the participant hasn't produced anything for this stage
    # in the current runtime — otherwise we'd clobber their work.
    if any(os.scandir(dst)):
        print(f"skip stage_0{n} — already has files (keeping your work)")
        continue
    for src_file in glob.glob(f"{src}/*"):
        shutil.copy(src_file, dst)
    print(f"seeded stage_0{n}/data from reference_outputs")


## 2. Spec — paste this into Gemini

Open the Gemini side panel in Colab (sparkles icon, top right) and paste the block below as your prompt. Then iterate.

```
For every PMID in `stage_02/data/screened.json` with verdict='include',
emit ONE artifact under `stage_03/data/`:

  PMC<id>.pdf   ← OA PDF when available
  PMC<id>.xml   ← JATS XML fallback
  <pmid>.json   ← Stage 1 metadata fallback (no fulltext available)

Also write `stage_03/data/fetched.json` mapping PMID → path
(no nulls — every included PMID gets at least metadata).

Pipeline:

  1. PMID → PMCID via NCBI ID Converter
     (https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/).
  2. PMCID → OA PDF URL via NCBI's PMC OA service
     (https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi).
  3. If no PDF (or PDF download fails), fall back to JATS XML via
     efetch.fcgi?db=pmc&id=<numeric>.
  4. If both fail, write the stage_01 metadata record as
     `stage_03/data/<pmid>.json`.
```


## 3. Gotchas Gemini probably won't know

Copy any that apply into Gemini if it goes off-track:

- **THE BIG ONE.** `oa.fcgi` returns FTP URLs like
  `ftp://ftp.ncbi.nlm.nih.gov/pub/pmc/oa_pdf/<dir>/<file>.pdf`. NCBI
  moved these files to
  `https://ftp.ncbi.nlm.nih.gov/pub/pmc/deprecated/oa_pdf/...` in April
  2026 *without updating oa.fcgi's response*. You MUST rewrite the URL
  before downloading. Gemini will not know this.
- **Don't use EuropePMC's getPdf** — currently HTTP 500.
- **Don't use `pmc.ncbi.nlm.nih.gov/articles/.../pdf/`** — gated by a
  JS proof-of-work; scripted clients get an HTML page, not a PDF.
- **Verify downloaded bytes.** A PDF starts with `%PDF`; a JATS XML
  body contains `<article` within the first few KB. Reject responses
  that don't match.
- **Be polite.** `time.sleep(0.4)` between NCBI calls.


## 4. Seed — a few lines to anchor Gemini in the right direction


In [ ]:
import json, os, time, urllib.request
import xml.etree.ElementTree as ET

STAGE = "stage_03"
DATA = f"{STAGE}/data"
os.makedirs(DATA, exist_ok=True)
HEADERS = {"User-Agent": "ar-bic-2026/0.1"}

with open("stage_02/data/screened.json") as f:
    included = [r for r in json.load(f) if r["verdict"] == "include"]
with open("stage_01/data/pmids.json") as f:
    metadata_by_pmid = {r["pmid"]: r for r in json.load(f)["records"]}


## 5. Your implementation

Drive Gemini to fill this in. Iterate until the verification cell below passes.


In [ ]:
# TODO: implement Stage 3 here.
# Read the spec above. Use the seed cell's imports.
# When done, run the verification cell next.


## 6. Verify


In [ ]:
import json, os
fetched = json.load(open("stage_03/data/fetched.json"))
for pmid, path in fetched.items():
    assert os.path.exists(path), f"{pmid}: missing {path}"
    floor = 100 if path.endswith(".json") else 10_000
    assert os.path.getsize(path) > floor, f"{pmid}: too small ({path})"
n_pdf = sum(1 for v in fetched.values() if v.endswith(".pdf"))
n_xml = sum(1 for v in fetched.values() if v.endswith(".xml"))
n_meta = sum(1 for v in fetched.values() if v.endswith(".json"))
print(f"OK — {n_pdf} PDF + {n_xml} XML + {n_meta} metadata fallback")


## 7. Run the eval grader

The eval reads only your stage's output and writes `stage_03/eval/eval_*.json` + `score.json`.


In [ ]:
!python eval/eval_03_script.py


**Note:** The reference Stage 3 output ships only the XML + metadata-only JSON fallbacks (no PDFs — too large to commit). Stage 4 handles all three input shapes, so the workshop still works end-to-end without the PDFs.


## 8. Stuck? Skip this stage

Copy the reference run's Stage 3 output into place so the next stage's notebook can still run. Use this sparingly — the point of the workshop is to *re-create* each stage.


In [ ]:
import os, shutil, glob
os.makedirs("stage_03/data", exist_ok=True)
for src in glob.glob("reference_outputs/stage_03/data/*"):
    shutil.copy(src, "stage_03/data/")
print("copied reference Stage 3 output:")
print(sorted(os.listdir("stage_03/data")))
